# MinIO et MongoDB, en direct

In [ ]:
# %pip install --quiet boto3 pyarrow duckdb pymongo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: c:\Users\Utilisateur\.pyenv\pyenv-win\versions\3.14.2\python.exe -m pip install --upgrade pip


In [ ]:
## On lance les deux conteneurs

In [ ]:
import subprocess, time, io, json

# on lance MinIO et MongoDB dans des conteneurs
subprocess.run("docker rm -f demo_minio demo_mongo".split(), capture_output=True)
subprocess.run(
    "docker run -d --name demo_minio -p 9000:9000 "
    "-e MINIO_ROOT_USER=minioadmin -e MINIO_ROOT_PASSWORD=minioadmin "
    "minio/minio server /data".split(),
    check=True,
)
subprocess.run(
    "docker run -d --name demo_mongo -p 27017:27017 mongo".split(), check=True
)
time.sleep(
    10
)  # on laisse les serveurs démarrer, utiliser une plus grande valeur que 10 si nécessaire

In [5]:
import boto3
from botocore.config import Config

# client S3 pointé sur MinIO ; path-style obligatoire, sinon boto3 vise un sous-domaine par bucket
s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
    region_name="us-east-1",
    config=Config(s3={"addressing_style": "path"}),
)
s3.create_bucket(Bucket="raw")
s3.list_buckets()["Buckets"]

[{'Name': 'raw',
  'CreationDate': datetime.datetime(2026, 8, 28, 9, 46, 22, 83000, tzinfo=tzutc())}]

## 1. Stockage objet : Parquet vers Bucket (via un 'in memoryy file')

In [6]:
import pyarrow as pa
import pyarrow.parquet as pq

# petit échantillon d'arrêts de tram (i.e. fait office d'extration GTFS)
arrets = [
    {
        "stop_id": "IDFM:490920",
        "route": "T3a",
        "stop_name": "Porte de Vincennes",
        "lat": 48.8470,
        "lon": 2.4103,
    },
    {
        "stop_id": "IDFM:463154",
        "route": "T3a",
        "stop_name": "Montempoivre",
        "lat": 48.8419,
        "lon": 2.4048,
    },
    {
        "stop_id": "IDFM:463155",
        "route": "T3b",
        "stop_name": "Porte de Pantin",
        "lat": 48.8886,
        "lon": 2.3932,
    },
]

# on écrit le Parquet en mémoire, puis on le dépose sur MinIO
buf = io.BytesIO()
pq.write_table(pa.Table.from_pylist(arrets), buf)
s3.put_object(Bucket="raw", Key="arrets_tram.parquet", Body=buf.getvalue())
s3.list_objects_v2(Bucket="raw")["Contents"]

[{'Key': 'arrets_tram.parquet',
  'LastModified': datetime.datetime(2026, 8, 28, 10, 7, 31, 994000, tzinfo=tzutc()),
  'ETag': '"5cb2f78e2adc0150ae4aba3c178a4e8b"',
  'Size': 1653,
  'StorageClass': 'STANDARD'}]

In [7]:
import duckdb

# DuckDB lit directement le Parquet depuis MinIO, sans le télécharger
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("""
    SET s3_endpoint='localhost:9000';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
""")
con.sql(
    "SELECT route, count(*) AS n_arrets FROM 's3://raw/arrets_tram.parquet' GROUP BY route ORDER BY route"
).df()

,route,n_arrets
0,T3a,2
1,T3b,1


## 2. NoSQL : exemple de documents imbriqués dans MongoDB

In [8]:
import pymongo

# on relit le Parquet depuis MinIO
data = s3.get_object(Bucket="raw", Key="arrets_tram.parquet")["Body"].read()
records = pq.read_table(io.BytesIO(data)).to_pylist()

# on dénormalise : un document par route, ses arrêts imbriqués (le geste clé du modèle document)
routes = {}
for r in records:
    routes.setdefault(r["route"], []).append(
        {
            "stop_id": r["stop_id"],
            "stop_name": r["stop_name"],
            "lat": r["lat"],
            "lon": r["lon"],
        }
    )
docs = [{"route": route, "stops": stops} for route, stops in routes.items()]
print(json.dumps(docs[0], indent=2, ensure_ascii=False))

{
  "route": "T3a",
  "stops": [
    {
      "stop_id": "IDFM:490920",
      "stop_name": "Porte de Vincennes",
      "lat": 48.847,
      "lon": 2.4103
    },
    {
      "stop_id": "IDFM:463154",
      "stop_name": "Montempoivre",
      "lat": 48.8419,
      "lon": 2.4048
    }
  ]
}


In [9]:
client = pymongo.MongoClient("mongodb://localhost:27017")
col = client["datalake"]["routes"]
col.delete_many({})
col.insert_many(docs)

# find : la route T3a avec ses arrêts, sans jointure
col.find_one({"route": "T3a"}, {"_id": 0})

{'route': 'T3a',
 'stops': [{'stop_id': 'IDFM:490920',
   'stop_name': 'Porte de Vincennes',
   'lat': 48.847,
   'lon': 2.4103},
  {'stop_id': 'IDFM:463154',
   'stop_name': 'Montempoivre',
   'lat': 48.8419,
   'lon': 2.4048}]}

In [10]:
# aggregate : nombre d'arrêts par route
list(
    col.aggregate(
        [
            {"$project": {"_id": 0, "route": 1, "n_arrets": {"$size": "$stops"}}},
            {"$sort": {"route": 1}},
        ]
    )
)

[{'route': 'T3a', 'n_arrets': 2}, {'route': 'T3b', 'n_arrets': 1}]

## Nettoyage

In [11]:
client.close()
subprocess.run("docker rm -f demo_minio demo_mongo".split(), capture_output=True)

CompletedProcess(args=['docker', 'rm', '-f', 'demo_minio', 'demo_mongo'], returncode=0, stdout=b'demo_minio\ndemo_mongo\n', stderr=b'')